In [0]:
from pyspark.sql.functions import col,current_timestamp,input_file_name
import re

In [0]:
display(dbutils.fs.ls("/Volumes/investment_pyspark/bronze/landing/tradebook/tradebook-IJV008-EQ_2_FY_24_25.csv"))

In [0]:
%sql
select * from read_files("/Volumes/investment_pyspark/bronze/landing/tradebook/")

In [0]:
#1 source path
source_path = "/Volumes/investment_pyspark/bronze/landing/tradebook/"

schema_path = "/Volumes/investment_pyspark/bronze/landing/tradebook/metadata/tradebook_schemas/"
checkpoint_path = "/Volumes/investment_pyspark/bronze/landing/tradebook/metadata/_checkpoints/"
df_bronze = (spark.readStream.format("cloudFiles") 
  .option("cloudFiles.format", "csv") 
  .option("cloudFiles.schemaLocation", schema_path) 
  .option("pathGlobFilter", "*.csv") 
  .option("header", True) 
  .load(source_path)
)
df_transformed = df_bronze.withColumn(
    "_inserted_timestamp", current_timestamp()
).withColumn("_source_file", col("_metadata.file_path"))
#display(df_transformed)
query = (
  df_transformed.writeStream
  .format("delta")
  .option("checkpointLocation", checkpoint_path)
  .outputMode("append")
  .trigger(availableNow=True)
  .table("investment_pyspark.bronze.tradebook_raw")
)
print(f"Streaming query started: {query.id}")
query.awaitTermination()
print(f"Stream completed: {query.lastProgress}")

In [0]:
%sql
select * from investment_pyspark.bronze.tradebook_raw;